In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("trading_sentiment_platform").getOrCreate()

In [0]:
filings_df = spark.read.table("workspace.sec_filings.stg_clean_10Ks")

display(filings_df)

### Create embedding

In [0]:
from pyspark.sql import functions as F

# select data and format sections to be embedded
sections_emb = (
    filings_df
    .select(
        "cik", "company_name", "tickers",
        "filing_date", "accessionNumber",
        "market_cap", "sector",
        F.col("bus_text").alias("BUSINESS"),
        F.col("risks_text").alias("RISKS"),
        F.col("mda_text").alias("MDA")
    )
    .selectExpr(
        "cik", "company_name", "tickers",
        "filing_date", "accessionNumber",
        "market_cap", "sector",
        "stack(3, 'BUSINESS', BUSINESS, 'RISKS', RISKS, 'MDA', MDA) as (section_name, section_text)"
    )
    .where("section_text is not null and length(trim(section_text)) > 2000")
    .withColumn("section_text_trimmed",
                F.substring("section_text", 1, 20000))
)
display(sections_emb)

In [0]:
# install openai
%pip install openai

In [0]:
# restart the Python kernel
%restart_python

In [0]:
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType
import json

# get the openai api key
api_key = dbutils.secrets.get("openai", "openai-api-key")

# choose the embedding model
EMBED_MODEL = "text-embedding-3-large"

# function to embed the financial text sections
def embed_text(text: str):
    if text is None or text.strip() == "":
        return None
    client = OpenAI(api_key = api_key)
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp.data[0].embedding

# apply udf on embed_text function for Spark
embed_udf = F.udf(embed_text, ArrayType(DoubleType()))

# embedding
embedded = (
    sections_emb
    .withColumn("embedding", embed_udf("section_text_trimmed"))
)

display(embedded)

# save embedding data
(embedded
 .write
 .mode("overwrite")
 .saveAsTable("sec_filings.fact_section_embeddings"))

### Compute risk drift

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np

# load fact_embedding data for risks only
risk_emb = (spark.read.table("workspace.sec_filings.fact_section_embeddings")
            .where(F.col("section_name") == "RISKS")
)

# window function to group by companies' CIKs and order by filing_date
w = Window.partitionBy("cik").orderBy(F.desc("market_cap"), F.desc("filing_date"))

# new dataframe that contains new columns with previous embeddings and date to the actual date
risk_prev = (risk_emb
             .withColumn("prev_embedding", F.lead("embedding").over(w))
             .withColumn("prev_filing_date", F.lead("filing_date").over(w))
             .where("prev_embedding is not null")
             )

# function that computes the cosine distance between risk embeddings
def cosine_distance(a, b):
    if a is None or b is None:
        return None
    
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return None
    return float(1.0 - np.dot(a, b) / denom)

# create a cosine_udf function for spark
cosine_udf = F.udf(cosine_distance, DoubleType())

# get the cosine distance in a new column
risk_drift = (risk_prev
            .withColumn("risk_cosine_distance", F.round(cosine_udf("embedding", "prev_embedding"), 2))
            )

# display(risk_emb)
display(risk_drift)

# save the risk_drift data in a fact table (keep only a few needed columns)
(risk_drift
 .select("cik","company_name","tickers",
         "filing_date","prev_filing_date",
         "risk_cosine_distance")
 .write
 .mode("overwrite")
 .saveAsTable("sec_filings.fact_risk_drift"))

In [0]:
high_risk_drift = (risk_drift
                   .where("risk_cosine_distance >=.2"))
display(high_risk_drift)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np

# load fact_embedding data for business only
bus_emb = (spark.read.table("workspace.sec_filings.fact_section_embeddings")
            .where(F.col("section_name") == "BUSINESS")
)

# window function to group by companies' CIKs and order by filing_date
w = Window.partitionBy("cik").orderBy(F.desc("market_cap"), F.desc("filing_date"))

# new dataframe that contains new columns with previous embeddings and date to the actual date
bus_prev = (bus_emb
             .withColumn("prev_embedding", F.lead("embedding").over(w))
             .withColumn("prev_filing_date", F.lead("filing_date").over(w))
             .where("prev_embedding is not null")
             )

# function that computes the cosine distance between risk embeddings
def cosine_distance(a, b):
    if a is None or b is None:
        return None
    
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return None
    return float(1.0 - np.dot(a, b) / denom)

# create a cosine_udf function for spark
cosine_udf = F.udf(cosine_distance, DoubleType())

# get the cosine distance in a new column
bus_drift = (bus_prev
            .withColumn("bus_cosine_distance", F.round(cosine_udf("embedding", "prev_embedding"), 2))
            )

# display(risk_emb)
display(bus_drift.select("cik","company_name","tickers",
         "filing_date","prev_filing_date",
         "bus_cosine_distance")
        .where("bus_cosine_distance>=.2"))

# save the risk_drift data in a fact table (keep only a few needed columns)
# (bus_drift
#  .select("cik","company_name","tickers",
#          "filing_date","prev_filing_date",
#          "bus_cosine_distance")
#  .write
#  .mode("overwrite")
#  .saveAsTable("sec_filings.fact_bus_drift"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np

# load fact_embedding data for business only
mda_emb = (spark.read.table("workspace.sec_filings.fact_section_embeddings")
            .where(F.col("section_name") == "MDA")
)

# window function to group by companies' CIKs and order by filing_date
w = Window.partitionBy("cik").orderBy(F.desc("market_cap"), F.desc("filing_date"))

# new dataframe that contains new columns with previous embeddings and date to the actual date
mda_prev = (mda_emb
             .withColumn("prev_embedding", F.lead("embedding").over(w))
             .withColumn("prev_filing_date", F.lead("filing_date").over(w))
             .where("prev_embedding is not null")
             )

# function that computes the cosine distance between risk embeddings
def cosine_distance(a, b):
    if a is None or b is None:
        return None
    
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return None
    return float(1.0 - np.dot(a, b) / denom)

# create a cosine_udf function for spark
cosine_udf = F.udf(cosine_distance, DoubleType())

# get the cosine distance in a new column
mda_drift = (mda_prev
            .withColumn("mda_cosine_distance", F.round(cosine_udf("embedding", "prev_embedding"), 2))
            )

# display(risk_emb)
display(mda_drift.select("cik","company_name","tickers",
         "filing_date","prev_filing_date",
         "mda_cosine_distance")
        .where("mda_cosine_distance>=.2"))

# save the risk_drift data in a fact table (keep only a few needed columns)
# (bus_drift
#  .select("cik","company_name","tickers",
#          "filing_date","prev_filing_date",
#          "bus_cosine_distance")
#  .write
#  .mode("overwrite")
#  .saveAsTable("sec_filings.fact_bus_drift"))

## Make the market reaction table

#### function that gets the market reaction

In [0]:
!pip install yfinance

In [0]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import timedelta
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import *
import pyspark.sql.functions as F

# function that gets the market reaction
def get_market_reaction(ticker: str, filing_date: str):
    try:
        # get the filing date in a pandas object
        filing_dt = pd.to_datetime(filing_date)

        start = filing_dt
        end   = filing_dt + timedelta(days=40)

        # download a market data from yahoo in a dataframe
        yahoo_df = yf.download(ticker, start=start, end=end, progress=False)
    
        if yahoo_df.empty:
            return [None, None, None, None]

        # sort row by index
        yahoo_df = yahoo_df.sort_index()
        prices = yahoo_df["Close"]

        # find the first available trading day at or after the filing date
        idx = prices.index.searchsorted(filing_dt)
        if idx >= len(prices):
            return [None, None, None, None]

        p0 = prices.iloc[idx]

        # helper function that returns the market price on a given day
        def safe_return(offset):
            tgt = idx + offset
            return float(prices.iloc[tgt] / p0 - 1) if tgt < len(prices) else None

        r1 = safe_return(1)
        r5 = safe_return(5)
        r20 = safe_return(20)

        # get volatility
        if len(prices[idx: idx+21]) >= 2:
            logrets = np.log(prices[idx: idx+21] / prices[idx: idx+21].shift(1)).dropna()
            vol20 = float(logrets.std() * np.sqrt(252))
        else:
            vol20 = None

        return [r1, r5, r20, vol20]

    except:
        return [None, None, None, None]
    


#### Build the pandas UDF for spark 

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType
# create a schema
schema = StructType([
    StructField("return_1d", DoubleType()),
    StructField("return_5d", DoubleType()),
    StructField("return_20d", DoubleType()),
    StructField("volatility_20d", DoubleType())
])

# the UDF function for Spark
@pandas_udf(schema)
def market_udf(ticker, fdate):
    results = [get_market_reaction(t, d) for t, d in zip(ticker, fdate)]
    return pd.DataFrame(results)


#### Apply the UDF and save market reaction data

In [0]:
import pyspark.sql.functions as F 

# load the stg_clean_10Ks data
filings = (spark.read.table("workspace.sec_filings.stg_clean_10Ks")
              .select("cik", "tickers", "filing_date", "market_cap")
              .distinct()
              .orderBy(F.desc("market_cap")))

# create a market dataframe with the market metrics
market = filings.withColumn(
    "market_metrics",
    market_udf("tickers", "filing_date")
).select(
    "cik",
    "tickers",
    "filing_date",
    F.round("market_metrics.return_1d", 2).alias("return_1d"),
    F.round("market_metrics.return_5d", 2).alias("return_5d"),
    F.round("market_metrics.return_20d", 2).alias("return_20d"),
    F.round("market_metrics.volatility_20d", 2).alias("volatility_20d")
)

display(market)

In [0]:
# save the market reaction data in a table
(market
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("sec_filings.fact_market_reaction"))

In [0]:
# def market_data(ticker, filing_date):
#         filing_dt = pd.to_datetime(filing_date)

#         start = filing_dt
#         end   = filing_dt + timedelta(days=40)

#         # download a market data from yahoo in a dataframe
#         yahoo_df = yf.download(ticker, start=start, end=end, progress=False)

#         if yahoo_df.empty:
#             return None, None

#         # sort row by index
#         yahoo_df = yahoo_df.sort_index()
#         prices = yahoo_df["Close"]

#         # find the first available trading day at or after the filing date
#         idx = prices.index.searchsorted(filing_dt)
#         if idx >= len(prices):
#             return None, None

#         p0 = prices.iloc[idx]

#         return yahoo_df, prices, idx
    
# file_yf = market_data("NVDA", "2024-02-21")
# file_yf
